In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
bronze_customermgmt=f"{catalog_name}.bronze.customermgmt"
silver_customer=f"{catalog_name}.silver.customer"
silver_account=f"{catalog_name}.silver.account"

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:
if str(batch_id) != "1":
    print("Batch 2/3 detected. CustomerMgmt.xml is a Batch 1 only file.")
    dbutils.notebook.exit("Skipped: Notebook only applicable for Batch 1")

In [0]:
df_bronze=spark.read.table(bronze_customermgmt).filter(col("_batch")==batch_id)

In [0]:
if batch_id=="1":
   try:
        df_customer_mgmt=df_bronze
        # Filter By Action Type
        df_customer_mgmt=df_customer_mgmt.filter(col("ActionType").isin(["NEW", "UPDCUST", "INACT"]))

        #Type Casting and Phone formatting
        df_customer_mgmt=df_customer_mgmt.withColumn("ActionTS",col("ActionTS").cast( "timestamp"))\
            .withColumn("C_ID",col("C_ID").cast("BIGINT"))\
            .withColumn("CA_ID",col("CA_ID").cast("BIGINT"))\
            .withColumn("C_TIER",expr("try_cast(C_TIER as TINYINT)"))\
            .withColumn("C_DOB",col("C_DOB").cast("DATE"))\
            .withColumn("Phone1", concat_ws("-",when(col("C_CTRY_1") != "", col("C_CTRY_1")),
                                                when(col("C_AREA_1") != "", col("C_AREA_1")),
                                                when(col("C_LOCAL_1") != "", col("C_LOCAL_1")),
                                                when(col("C_EXT_1") != "", col("C_EXT_1"))
                                                ))\
            .withColumn("Phone2", concat_ws("-",when(col("C_CTRY_2") != "", col("C_CTRY_2")),
                                                when(col("C_AREA_2") != "", col("C_AREA_2")),
                                                when(col("C_LOCAL_2") != "", col("C_LOCAL_2")),
                                                when(col("C_EXT_2") != "", col("C_EXT_2"))
                                                ))\
            .withColumn("Phone3", concat_ws("-",when(col("C_CTRY_3") != "", col("C_CTRY_3")),
                                                when(col("C_AREA_3") != "", col("C_AREA_3")),
                                                when(col("C_LOCAL_3") != "", col("C_LOCAL_3")),
                                                when(col("C_EXT_3") != "", col("C_EXT_3"))
                                                ))
        #Added Status Column
        df_customer_mgmt=df_customer_mgmt.withColumn("Status",
                                                    when(col("ActionType")=="INACT",lit("INAC")).otherwise(lit("ACTV")))
        df_customer_mgmt=df_customer_mgmt.drop("C_CTRY_1","C_AREA_1","C_LOCAL_1","C_EXT_1","C_CTRY_2","C_AREA_2","C_LOCAL_2","C_EXT_2","C_CTRY_3","C_AREA_3","C_LOCAL_3","C_EXT_3","_ingest_ts","_source_file","CA_ID","CA_TAX_ST","CA_B_ID","CA_NAME","ActionType")

        #added audit Columns
        df_customer_mgmt=df_customer_mgmt.withColumn("_load_ts",current_timestamp())
        
        # df_customer_mgmt.limit(10).display()
        # df_customer_mgmt.select*("ActionType").distinct().display()
        # df_customer_mgmt.select("Status").distinct().display()

        #Deduplication
        df_customer_mgmt.createOrReplaceTempView("v_customer_mgmt")
        df_customer_mgmt=spark.sql("""
                        SELECT * EXCEPT (rn) 
                        from (
                            SELECT * ,row_number() over(
                                partition by C_ID
                                order by ActionTS desc
                            ) As rn
                            FROM v_customer_mgmt
                        ) where rn=1
        """)
        # df_customer_mgmt.limit(10).display()
        print(f"Writing to Table {silver_customer}")
        df_customer_mgmt.write.format("delta")\
            .partitionBy("_batch")\
            .option("mergeSchema","true")\
            .mode("append")\
            .saveAsTable(f"{silver_customer}")
        count=spark.table(silver_customer).count()
        print(f"written successfully with row {count}")


        run_id=df_customer_mgmt.select("_run_id").first()[0]
        cust_history = spark.sql(f"DESCRIBE HISTORY {silver_customer}").first()
        cust_rows = int(cust_history["operationMetrics"].get("numOutputRows", 0))

        log_pipeline_recon(
            spark=spark,
            run_id=run_id,
            batch_id=batch_id,
            domain="CUSTOMER",
            table_name="customer",
            source_layer="bronze",
            target_layer="silver",
            source_count=cust_rows,
            target_count=cust_rows
        )
   except Exception as e:
       raise e
    
    

In [0]:
if batch_id =="1":
    try:
        df_account=df_bronze.filter(col("ActionType").isin("ADDACCT", "UPDACCT", "CLOSEACCT", "NEW"))
        df_account=df_account.withColumn("ActionTS",col("ActionTS").cast( "timestamp"))\
            .withColumn("CA_ID",col("CA_ID").cast("bigint"))\
            .withColumn("C_ID",col("C_ID").cast("bigint"))\
            .withColumn("CA_TAX_ST",expr("try_cast(CA_TAX_ST as TINYINT)"))\
            .withColumn("CA_B_ID",col("CA_B_ID").cast("bigint"))
        
        df_account=df_account.withColumn("Status",
                when(col("ActionType") == "CLOSEACCT", lit("INAC"))
                .otherwise(lit("ACTV"))) 

        #deduplication
        df_account.createOrReplaceTempView("v_account")
        df_account=spark.sql("""
            SELECT * EXCEPT (rn) 
            from (
                SELECT *,row_number() over(
                    partition by CA_ID
                    order by ActionTS desc
                ) As rn
                FROM v_account
            ) where rn=1
        """)

        run_id=df_account.select("_run_id").first()[0]
        df_account = df_account.withColumn("_load_ts", current_timestamp())
        df_account=df_account.drop("_ingest_ts", "_source_file", "ActionType")

        
        # df_account.limit(10).display()   
        # print(df_account.count())   
        print(f"Writing to Table {silver_account}") 
        df_account.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(silver_account)
        # count=spark.table(silver_account).count()
        # print(f"written successfully with row {count}")   

        acct_history = spark.sql(f"DESCRIBE HISTORY {silver_account}").first()
        acct_rows = int(acct_history["operationMetrics"].get("numOutputRows", 0))

        log_pipeline_recon(
            spark=spark,
            run_id=run_id,
            batch_id=batch_id,
            domain="CUSTOMER",
            table_name="account",
            source_layer="bronze",
            target_layer="silver",
            source_count=acct_rows,
            target_count=acct_rows
        )
        
        # Log Audit for Account
        log_audit_event(
            spark=spark,
            run_id=run_id,
            batch=batch_id,
            layer="silver",
            table_name="account",
            operation="OVERWRITE",
            rows_affected=acct_rows
        )
    except Exception as e:
       raise e
     